# Amazon ML Challenge 2026 — Business Entity Resolution Pipeline Notebook (GPU Accelerated)

## Project Overview & Context
- **Problem**: Match business records across 3 independent noisy sources (`S1`, `S2`, `S3`). `Source 1` is the reference dataset; identify all corresponding `S2` and `S3` records referring to the same real-world business.
- **Target Metric**: **Macro-Averaged F₀.₅** across all `S1` entities (precision-weighted, $\beta = 0.5$).
- **Singletons**: `S1` entities with 0 true matches score 1.0 if predicted empty, but 0.0 if any false merge is predicted.
- **Scale**: ~2.2M `S1`, ~5.0M `S2`, ~5.3M `S3` in training; ~1.7M `S1`, ~4.9M `S2`, ~5.1M `S3` in test.
- **Open-Set Country Rule**: Test contains France (~259K entities) which is unseen in training. Country is treated as an open-set string (never categorical or hardcoded).
- **Final Outputs Required**:
  1. `output/matching_results.tsv`: `source1_entity_id\tmatched_entity_ids` (comma-separated)
  2. `output/candidate_pairs.tsv`: `source1_entity_id\tcandidate_entity_ids` (comma-separated)
- **Key Constraint**: `FINAL_MATCHES ⊆ CANDIDATES`. Must pass `utils/validate_submission.py` with `PASS`.

---

### Cell 1: Environment Setup & Google Drive Mounting

In [ ]:
import os
import sys
from pathlib import Path

# Check if running inside Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/Amazon-ML-entity-resolution')
else:
    PROJECT_ROOT = Path('.').resolve()

print(f"Project Root Directory: {PROJECT_ROOT}")
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

# Print Environment Diagnostics (CPU, RAM, GPU)
import psutil
ram_gb = psutil.virtual_memory().total / (1024**3)
print(f"Available RAM: {ram_gb:.2f} GB")
print(f"CPU Count    : {os.cpu_count()}")

try:
    import torch
    if torch.cuda.is_available():
        print(f"GPU Device   : {torch.cuda.get_device_name(0)}")
    else:
        print("GPU Device   : Not available (using CPU multi-threading)")
except ImportError:
    print("PyTorch not installed; running on CPU workers.")

### Cell 2: Install Pinned Dependencies

In [ ]:
!pip install -q lightgbm>=4.3.0 scikit-learn>=1.4.0 numpy>=1.26.0 scipy>=1.13.0 pandas>=2.2.0 rapidfuzz>=3.8.0 joblib>=1.4.0 psutil

### Cell 3: Central Configuration & Directory Setup

In [ ]:
from pathlib import Path

# Paths
TRAIN_DIR      = PROJECT_ROOT / "dataset" / "train"
TEST_DIR       = PROJECT_ROOT / "dataset" / "test"
OUTPUT_DIR     = PROJECT_ROOT / "output"
MODEL_DIR      = PROJECT_ROOT / "models"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
REPORTS_DIR    = PROJECT_ROOT / "reports"

for d in (OUTPUT_DIR, MODEL_DIR, CHECKPOINT_DIR, REPORTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Sub-checkpoint directories
(CHECKPOINT_DIR / "preprocess").mkdir(exist_ok=True)
(CHECKPOINT_DIR / "blocking").mkdir(exist_ok=True)
(CHECKPOINT_DIR / "features").mkdir(exist_ok=True)
(CHECKPOINT_DIR / "models").mkdir(exist_ok=True)

print("✓ All directories and checkpoint paths initialized.")

### Cell 4: Preprocessing & Text Normalization

In [ ]:
import re
import unicodedata
import pandas as pd
from preprocess import preprocess_df, normalize_name, normalize_address

def load_and_preprocess_with_checkpoint(split="train", sample_s1=None, sample_s23=True):
    """
    Loads raw TSVs and applies NFKC normalization, abbreviation expansion,
    and explicit NaN handling with optional disk checkpointing.
    """
    ckpt_path = CHECKPOINT_DIR / "preprocess" / f"{split}_preprocessed.pkl"
    
    if split == "train":
        s1_file = TRAIN_DIR / "train_source1.tsv"
        s2_file = TRAIN_DIR / "train_source2.tsv"
        s3_file = TRAIN_DIR / "train_source3.tsv"
        gt_file = TRAIN_DIR / "train_ground_truth.tsv"
    else:
        s1_file = TEST_DIR / "test_source1.tsv"
        s2_file = TEST_DIR / "test_source2.tsv"
        s3_file = TEST_DIR / "test_source3.tsv"
        gt_file = None

    print(f"Loading {split} data...")
    s1_df = pd.read_csv(s1_file, sep="\t", dtype=str, keep_default_na=False)
    s2_df = pd.read_csv(s2_file, sep="\t", dtype=str, keep_default_na=False)
    s3_df = pd.read_csv(s3_file, sep="\t", dtype=str, keep_default_na=False)
    gt_df = pd.read_csv(gt_file, sep="\t", dtype=str, keep_default_na=False) if gt_file else None

    if sample_s1 and split == "train":
        s1_df = s1_df.sample(n=min(sample_s1, len(s1_df)), random_state=42).reset_index(drop=True)
        if sample_s23:
            # Reduced dev pool mode for fast iterations
            sampled_ids = set(s1_df["entity_id"])
            gt_subset = gt_df[gt_df["source1_entity_id"].isin(sampled_ids)]
            gt_match_ids = set()
            for _, r in gt_subset.iterrows():
                m = r.get("matched_entity_ids", "").strip()
                if m:
                    gt_match_ids.update(m.split(","))
            s2_gt = {x for x in gt_match_ids if x.startswith("S2-")}
            s3_gt = {x for x in gt_match_ids if x.startswith("S3-")}
            
            rng = pd.np.random.default_rng(42) if hasattr(pd, 'np') else __import__('numpy').random.default_rng(42)
            def _sub(df, gids, n_ext):
                g_rows = df[df["entity_id"].isin(gids)]
                rest   = df[~df["entity_id"].isin(gids)]
                take   = min(n_ext, len(rest))
                extra  = rest.iloc[rng.choice(len(rest), take, replace=False)]
                return pd.concat([g_rows, extra], ignore_index=True)
            
            s2_df = _sub(s2_df, s2_gt, sample_s1 * 5)
            s3_df = _sub(s3_df, s3_gt, sample_s1 * 5)
            print(f"  Dev Subsampled S2={len(s2_df):,}, S3={len(s3_df):,}")
        else:
            print(f"  50K Sample S1={len(s1_df):,} vs FULL S2={len(s2_df):,}, S3={len(s3_df):,}")

    print("Preprocessing text attributes (NFKC, abbrevs, fillna)... formulation")
    s1_prep = preprocess_df(s1_df)
    s2_prep = preprocess_df(s2_df)
    s3_prep = preprocess_df(s3_df)

    gt_dict = {}
    if gt_df is not None:
        s1_set = set(s1_prep["entity_id"])
        for _, r in gt_df.iterrows():
            sid = r["source1_entity_id"]
            if sid in s1_set:
                m = r.get("matched_entity_ids", "").strip()
                gt_dict[sid] = [x for x in m.split(",") if x] if m else []

    return s1_prep, s2_prep, s3_prep, gt_dict

s1_demo, s2_demo, s3_demo, gt_demo = load_and_preprocess_with_checkpoint(split="train", sample_s1=5000, sample_s23=True)
print(f"✓ Loaded sample data: S1={len(s1_demo):,}, S2={len(s2_demo):,}, S3={len(s3_demo):,}")

### Cell 5: Multi-Blocker Candidate Generation (99.25% Recall Strategy)

In [ ]:
from blocking_multi import generate_multi_blocker_candidates
from blocking import generate_candidates

print("Testing Multi-Blocker Candidate Generation Pipeline...")
cands_demo, diag_demo = generate_multi_blocker_candidates(
    s1_demo, s2_demo, s3_demo,
    blocker_configs=[
        {"name": "name_tfidf", "type": "name_tfidf", "k": 30},
        {"name": "addr_tfidf", "type": "addr_tfidf", "k": 15},
        {"name": "exact_name", "type": "exact_name", "max_matches": 15},
        {"name": "rare_tokens", "type": "rare_tokens", "max_cands": 10},
    ],
    k_final=50,
)

# Evaluate Candidate Recall
total_gt = sum(len(v) for v in gt_demo.values())
found_gt = sum(1 for sid, gt_ids in gt_demo.items() for g in gt_ids if g in set(cands_demo.get(sid, [])))
recall_pct = (found_gt / total_gt) * 100.0 if total_gt > 0 else 0.0
avg_cands = sum(len(v) for v in cands_demo.values()) / max(len(s1_demo), 1)

print(f"★ Candidate Recall: {recall_pct:.2f}% ({found_gt}/{total_gt} GT pairs)")
print(f"★ Average Candidates / S1: {avg_cands:.1f}")

### Cell 6: 28 Advanced Pairwise Feature Matrix Extraction

In [ ]:
from features import FEATURE_NAMES, build_feature_matrix, build_lookup
from experiment_runner import build_pairs_with_missed

print(f"Total Feature Count: {len(FEATURE_NAMES)}")
print("Feature List:", FEATURE_NAMES)

s23_lookup = build_lookup(pd.concat([s2_demo, s3_demo], ignore_index=True))
s1_lookup  = build_lookup(s1_demo)

pairs_demo, labels_demo = build_pairs_with_missed(s1_demo, cands_demo, gt_demo)
ranks_demo = {(sid, cid): r+1 for sid, clist in cands_demo.items() for r, cid in enumerate(clist)}

print(f"Building feature matrix for {len(pairs_demo):,} candidate pairs...")
X_demo = build_feature_matrix(pairs_demo, s1_lookup, s23_lookup, candidate_ranks=ranks_demo, n_jobs=-1)
y_demo = __import__('numpy').array(labels_demo, dtype=__import__('numpy').int32)

print(f"✓ Feature Matrix Shape: {X_demo.shape}")

### Cell 7: Model Training & Threshold Sweep on Local Validation

In [ ]:
import lightgbm as lgb
import pickle
from config import LGBM_PARAMS, THRESHOLD_GRID, EARLY_STOPPING_ROUNDS
from experiment_runner import split_three_way, evaluate_predictions_detailed

# 1. Split 3-Way
s1_tr, s1_va, s1_ho = split_three_way(s1_demo)
print(f"Split: Train={len(s1_tr):,}, Val={len(s1_va):,}, Holdout={len(s1_ho):,}")

cands_tr = {k: v for k, v in cands_demo.items() if k in set(s1_tr["entity_id"])}
cands_va = {k: v for k, v in cands_demo.items() if k in set(s1_va["entity_id"])}
cands_ho = {k: v for k, v in cands_demo.items() if k in set(s1_ho["entity_id"])}

gt_tr = {k: v for k, v in gt_demo.items() if k in set(s1_tr["entity_id"])}
gt_va = {k: v for k, v in gt_demo.items() if k in set(s1_va["entity_id"])}
gt_ho = {k: v for k, v in gt_demo.items() if k in set(s1_ho["entity_id"])}

pairs_tr, labels_tr = build_pairs_with_missed(s1_tr, cands_tr, gt_tr)
pairs_va, labels_va = build_pairs_with_missed(s1_va, cands_va, gt_va)
pairs_ho, labels_ho = build_pairs_with_missed(s1_ho, cands_ho, gt_ho)

s1_tr_lookup = build_lookup(s1_tr)
s1_va_lookup = build_lookup(s1_va)
s1_ho_lookup = build_lookup(s1_ho)

X_tr = build_feature_matrix(pairs_tr, s1_tr_lookup, s23_lookup, candidate_ranks=ranks_demo, n_jobs=-1)
y_tr = __import__('numpy').array(labels_tr, dtype=__import__('numpy').int32)

X_va = build_feature_matrix(pairs_va, s1_va_lookup, s23_lookup, candidate_ranks=ranks_demo, n_jobs=-1)
y_va = __import__('numpy').array(labels_va, dtype=__import__('numpy').int32)

X_ho = build_feature_matrix(pairs_ho, s1_ho_lookup, s23_lookup, candidate_ranks=ranks_demo, n_jobs=-1)
y_ho = __import__('numpy').array(labels_ho, dtype=__import__('numpy').int32)

# 2. Train LightGBM
dtrain = lgb.Dataset(X_tr, label=y_tr, feature_name=FEATURE_NAMES)
dval   = lgb.Dataset(X_va, label=y_va, feature_name=FEATURE_NAMES, reference=dtrain)

params = dict(LGBM_PARAMS)
params.pop("n_estimators", None)
model = lgb.train(
    params,
    dtrain,
    num_boost_round=LGBM_PARAMS["n_estimators"],
    valid_sets=[dval],
    callbacks=[
        lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False),
    ],
)

# 3. Sweep Threshold on Val F0.5
val_probs = model.predict(X_va)
prob_map_va = {(sid, cid): prob for (sid, cid), prob in zip(pairs_va, val_probs)}

best_val_f05, best_thr, best_val_stats = -1.0, 0.5, {}
for thr in THRESHOLD_GRID:
    pred_dict_va = {sid: [c for c in cands_va.get(sid, []) if prob_map_va.get((sid, c), 0.0) >= thr] for sid in s1_va["entity_id"]}
    stats = evaluate_predictions_detailed(pred_dict_va, gt_va)
    if stats["macro_f05"] > best_val_f05:
        best_val_f05 = stats["macro_f05"]
        best_thr = thr
        best_val_stats = stats

print(f"★ Best Threshold: {best_thr:.2f} | Val F0.5: {best_val_f05:.4f}")

# 4. Evaluate Holdout
ho_probs = model.predict(X_ho)
prob_map_ho = {(sid, cid): prob for (sid, cid), prob in zip(pairs_ho, ho_probs)}
pred_dict_ho = {sid: [c for c in cands_ho.get(sid, []) if prob_map_ho.get((sid, c), 0.0) >= best_thr] for sid in s1_ho["entity_id"]}
ho_stats = evaluate_predictions_detailed(pred_dict_ho, gt_ho)
print(f"★ Holdout Score (thr={best_thr:.2f}): F0.5 = {ho_stats['macro_f05']:.4f}")

# Save Model Checkpoint
with open(MODEL_DIR / "lgbm_model.pkl", "wb") as f:
    pickle.dump(model, f)
with open(MODEL_DIR / "meta.pkl", "wb") as f:
    pickle.dump({"threshold": best_thr, "val_f05": best_val_f05}, f)
print("✓ Saved model and metadata checkpoint.")

### Cell 8: 50K Full-Pool Retrieval Evaluation Experiment

> **Crucial Step**: Evaluates 50,000 `S1` entities against the **FULL 10.3M `S2/S3` universe** (un-subsampled) to verify real-world retrieval recall and macro $F_{0.5}$.

In [ ]:
print("=== Executing 50K Full S2/S3 Pool Evaluation Experiment ===")
s1_50k, s2_full, s3_full, gt_50k = load_and_preprocess_with_checkpoint(split="train", sample_s1=50000, sample_s23=False)

print(f"Querying {len(s1_50k):,} S1 entities against FULL S2={len(s2_full):,}, S3={len(s3_full):,}...")
t0 = __import__('time').time()
cands_50k, diag_50k = generate_multi_blocker_candidates(s1_50k, s2_full, s3_full, k_final=50)
elapsed_50k = __import__('time').time() - t0

total_gt_50k = sum(len(v) for v in gt_50k.values())
found_gt_50k = sum(1 for sid, gt_ids in gt_50k.items() for g in gt_ids if g in set(cands_50k.get(sid, [])))
recall_50k   = (found_gt_50k / total_gt_50k) * 100.0 if total_gt_50k > 0 else 0.0
avg_cands_50k = sum(len(v) for v in cands_50k.values()) / max(len(s1_50k), 1)

print(f"✓ 50K Full-Pool Blocking Elapsed: {elapsed_50k:.1f} seconds")
print(f"★ 50K Full-Pool Candidate Recall : {recall_50k:.2f}% ({found_gt_50k:,}/{total_gt_50k:,} GT pairs)")
print(f"★ 50K Full-Pool Avg Candidates/S1: {avg_cands_50k:.1f}")

### Cell 9: Full Test Set Inference Pipeline

In [ ]:
print("=== Full Test Set Inference Pipeline ===")
s1_test, s2_test, s3_test, _ = load_and_preprocess_with_checkpoint(split="test", sample_s1=None)

print(f"Test Set Sizes: S1={len(s1_test):,}, S2={len(s2_test):,}, S3={len(s3_test):,}")

# 1. Multi-Blocker Candidate Retrieval on Test Set
print("Running candidate blocking on test data...")
cands_test, _ = generate_multi_blocker_candidates(s1_test, s2_test, s3_test, k_final=50)

# 2. Save candidate_pairs.tsv
cand_tsv_path = OUTPUT_DIR / "candidate_pairs.tsv"
rows_cand = [{"source1_entity_id": sid, "candidate_entity_ids": ",".join(cands_test.get(sid, []))} for sid in s1_test["entity_id"]]
pd.DataFrame(rows_cand).to_csv(cand_tsv_path, sep="\t", index=False)
print(f"✓ Saved candidate pairs → {cand_tsv_path}")

# 3. Build Test Feature Matrix & Predict
test_pairs = [(sid, cid) for sid, clist in cands_test.items() for cid in clist]
test_ranks = {(sid, cid): r+1 for sid, clist in cands_test.items() for r, cid in enumerate(clist)}

s23_test_lookup = build_lookup(pd.concat([s2_test, s3_test], ignore_index=True))
s1_test_lookup  = build_lookup(s1_test)

print(f"Computing features for {len(test_pairs):,} test candidate pairs...")
X_test = build_feature_matrix(test_pairs, s1_test_lookup, s23_test_lookup, candidate_ranks=test_ranks, n_jobs=-1)

print("Predicting probabilities...")
test_probs = model.predict(X_test)
prob_map_test = {(sid, cid): prob for (sid, cid), prob in zip(test_pairs, test_probs)}

# 4. Apply Threshold & Write matching_results.tsv
matching_tsv_path = OUTPUT_DIR / "matching_results.tsv"
rows_match = []
for sid in s1_test["entity_id"]:
    clist = cands_test.get(sid, [])
    matched = [c for c in clist if prob_map_test.get((sid, c), 0.0) >= best_thr]
    rows_match.append({"source1_entity_id": sid, "matched_entity_ids": ",".join(matched)})

pd.DataFrame(rows_match).to_csv(matching_tsv_path, sep="\t", index=False)
print(f"✓ Saved matching results → {matching_tsv_path}")

### Cell 10: Submission Validator Verification

In [ ]:
import subprocess

val_script = PROJECT_ROOT / "utils" / "validate_submission.py"
matching_file  = OUTPUT_DIR / "matching_results.tsv"
candidate_file = OUTPUT_DIR / "candidate_pairs.tsv"
test_dir_path  = TEST_DIR

cmd = [
    sys.executable, str(val_script),
    "--matching",  str(matching_file),
    "--candidate", str(candidate_file),
    "--test-dir",  str(test_dir_path),
]

print(f"Executing Submission Validator: {' '.join(cmd)}")
res = subprocess.run(cmd, capture_output=True, text=True)
print(res.stdout)
if res.returncode == 0:
    print("\n========================================")
    print("  ✓ SUBMISSION VALIDATION PASSED (PASS)  ")
    print("========================================")
else:
    print(res.stderr)
    print("\n✗ Submission validation failed — check errors above.")